In [5]:
import pandas as pd
import numpy as np

# 读取播放日志 (tsv)
logs = pd.read_csv(
    'Last.fm Dataset/lastfm-dataset-1K/userid-timestamp-artid-artname-traid-traname.tsv',
    sep='\t', header=None,
    names=['user_id', 'timestamp','artist_id','artist_name','track_id','track_name'],
    on_bad_lines='skip',
    encoding='utf-8'
)

# 解析时间戳 (ISO 8601 格式，末尾 Z 表示 UTC)
logs['play_time'] = pd.to_datetime(logs['timestamp'], utc=True)
logs['date'] = logs['play_time'].dt.date  # 先保留为 date 对象，后面按需转

print(f"日志原始记录数: {len(logs)}")
print(f"唯一用户数: {logs['user_id'].nunique()}")
print(f"时间范围: {logs['date'].min()} ~ {logs['date'].max()}")

日志原始记录数: 19098853
唯一用户数: 992
时间范围: 2005-02-14 ~ 2013-09-29


In [6]:
# 读取用户画像 (tsv)
users = pd.read_csv(
    'Last.fm Dataset/lastfm-dataset-1K/userid-profile.tsv',
    sep='\t', header=None,
    names=['user_id', 'gender', 'age', 'country', 'signup'],
    on_bad_lines='skip',
    encoding='utf-8'
)

# 解析注册时间
users['signup_date'] = pd.to_datetime(users['signup'], errors='coerce')

print(f"用户画像行数: {len(users)}")
print(users.head())

用户画像行数: 993
       user_id  gender  age        country        signup signup_date
0          #id  gender  age        country    registered         NaT
1  user_000001       m  NaN          Japan  Aug 13, 2006  2006-08-13
2  user_000002       f  NaN           Peru  Feb 24, 2006  2006-02-24
3  user_000003       m   22  United States  Oct 30, 2005  2005-10-30
4  user_000004       f  NaN            NaN  Apr 26, 2006  2006-04-26


C:\Users\25776\AppData\Local\Temp\ipykernel_25956\3016267309.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  users['signup_date'] = pd.to_datetime(users['signup'], errors='coerce')


In [7]:
# 按照 README 截止日期截断
cutoff_date = pd.Timestamp('2009-05-05')
logs['date'] = pd.to_datetime(logs['date'])  # 确保为 Timestamp
logs_filtered = logs[logs['date'] <= cutoff_date].copy()

# 补充分析用字段
logs_filtered['hour'] = logs_filtered['play_time'].dt.hour
logs_filtered['weekday'] = logs_filtered['play_time'].dt.dayofweek  # 0=周一

print(f"过滤后记录数: {len(logs_filtered)}, 用户数: {logs_filtered['user_id'].nunique()}")
print(f"时间范围: {logs_filtered['date'].min()} ~ {logs_filtered['date'].max()}")

过滤后记录数: 18945380, 用户数: 991
时间范围: 2005-02-14 00:00:00 ~ 2009-05-05 00:00:00


In [8]:
# 只保留后续需要的列
logs_clean = logs_filtered[['user_id', 'date', 'play_time', 'hour', 'weekday']].copy()
users_clean = users.copy()

# 保存为 CSV
logs_clean.to_csv('logs_clean.csv', index=False)
users_clean.to_csv('users_clean.csv', index=False)

print("清洗数据已保存: logs_clean.csv, users_clean.csv")

清洗数据已保存: logs_clean.csv, users_clean.csv
